
# Advanced Transformer-Based rPPG Pipeline

This notebook contains a significantly upgraded rPPG pipeline with:

- MediaPipe FaceMesh ROI extraction
- Forehead + cheek extraction
- POS signal extraction
- Temporal differencing
- Improved preprocessing
- Stronger CNN encoder
- Transformer temporal modeling
- Spectral + Pearson hybrid loss
- Better BPM estimation
- Modern training pipeline

Designed for GPU training using PyTorch.


In [1]:

# Install dependencies (run once)

# !pip install torch torchvision torchaudio
# !pip install mediapipe opencv-python scipy numpy matplotlib tqdm


In [2]:

import os
import cv2
import math
import torch
import random
import mediapipe as mp
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from scipy.signal import butter, filtfilt
from torch.utils.data import Dataset, DataLoader


In [3]:

class CFG:

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    IMG_SIZE = 128

    SEQ_LEN = 300

    BATCH_SIZE = 2

    EPOCHS = 100

    LR = 1e-4

    D_MODEL = 128

    NHEAD = 8

    NUM_LAYERS = 4

cfg = CFG()

print(cfg.DEVICE)


cpu


## FaceMesh Initialization

In [4]:

mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)


In [5]:

FOREHEAD_IDXS = [10, 67, 103, 109, 338, 297]
LEFT_CHEEK    = [50, 187, 205, 207]
RIGHT_CHEEK   = [280, 425, 411, 427]


## ROI Extraction

In [6]:

def extract_roi(frame, landmarks, indices):

    h, w, _ = frame.shape

    pts = []

    for idx in indices:

        lm = landmarks.landmark[idx]

        pts.append((int(lm.x * w), int(lm.y * h)))

    pts = np.array(pts, dtype=np.int32)

    mask = np.zeros((h, w), dtype=np.uint8)

    cv2.fillConvexPoly(mask, pts, 255)

    roi = cv2.bitwise_and(frame, frame, mask=mask)

    x, y, bw, bh = cv2.boundingRect(pts)

    roi = roi[y:y+bh, x:x+bw]

    return roi


## Signal Processing

In [7]:

def POS_WANG(rgb_signals):

    rgb = np.asarray(rgb_signals)

    mean_rgb = np.mean(rgb, axis=0)

    rgb = rgb / (mean_rgb + 1e-8)

    projection = np.array([
        [0, 1, -1],
        [-2, 1, 1]
    ])

    S = projection @ rgb.T

    std1 = np.std(S[0])
    std2 = np.std(S[1])

    alpha = std1 / (std2 + 1e-8)

    h = S[0] + alpha * S[1]

    return h


In [8]:

def bandpass_filter(signal, fs=30):

    lowcut = 0.75
    highcut = 3.0

    nyquist = fs * 0.5

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(5, [low, high], btype='band')

    filtered = filtfilt(b, a, signal)

    filtered = (filtered - filtered.mean()) / (filtered.std() + 1e-8)

    return filtered


In [9]:

def compute_bpm(signal, fs=30):

    signal = bandpass_filter(signal, fs)

    freqs = np.fft.rfftfreq(len(signal), d=1/fs)

    fft = np.abs(np.fft.rfft(signal))

    mask = (freqs >= 0.75) & (freqs <= 3.0)

    freqs = freqs[mask]
    fft = fft[mask]

    bpm = freqs[np.argmax(fft)] * 60

    return bpm


## Temporal Difference

In [10]:

def temporal_difference(frames):

    diff_frames = []

    for i in range(1, len(frames)):

        diff = frames[i].astype(np.float32) - frames[i-1].astype(np.float32)

        diff_frames.append(diff)

    return np.array(diff_frames)


## Video Processing

In [11]:

def process_video(video_path):

    cap = cv2.VideoCapture(video_path)

    frames = []
    rgb_signals = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = face_mesh.process(frame)

        if results.multi_face_landmarks:

            landmarks = results.multi_face_landmarks[0]

            forehead = extract_roi(frame, landmarks, FOREHEAD_IDXS)
            left_cheek = extract_roi(frame, landmarks, LEFT_CHEEK)
            right_cheek = extract_roi(frame, landmarks, RIGHT_CHEEK)

            rois = [forehead, left_cheek, right_cheek]

            merged = []

            for roi in rois:

                if roi.size == 0:
                    continue

                roi = cv2.resize(roi, (cfg.IMG_SIZE, cfg.IMG_SIZE))

                merged.append(roi)

                rgb_mean = np.mean(roi.reshape(-1, 3), axis=0)

                rgb_signals.append(rgb_mean)

            if len(merged) > 0:

                face = np.mean(np.stack(merged), axis=0)

                frames.append(face)

    cap.release()

    frames = np.array(frames)

    frames = temporal_difference(frames)

    mean = np.mean(frames, axis=(0,1,2), keepdims=True)
    std = np.std(frames, axis=(0,1,2), keepdims=True)

    frames = (frames - mean) / (std + 1e-8)

    signal = POS_WANG(rgb_signals)

    signal = bandpass_filter(signal)

    return frames, signal


## Dataset

In [12]:

class RPPGDataset(Dataset):

    def __init__(self, video_paths):

        self.video_paths = video_paths

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):

        video_path = self.video_paths[idx]

        frames, signal = process_video(video_path)

        frames = torch.tensor(frames, dtype=torch.float32)
        signal = torch.tensor(signal[:len(frames)], dtype=torch.float32)

        return frames, signal


## Improved CNN Encoder

In [13]:

class ImprovedSpatialEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(128, cfg.D_MODEL)

    def forward(self, x):

        x = self.encoder(x)

        x = x.flatten(1)

        x = self.fc(x)

        return x


## Transformer Model

In [14]:

class RPPGTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.spatial = ImprovedSpatialEncoder()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.D_MODEL,
            nhead=cfg.NHEAD,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=cfg.NUM_LAYERS
        )

        self.temporal_conv = nn.Sequential(
            nn.Conv1d(cfg.D_MODEL, cfg.D_MODEL, 3, padding=1),
            nn.ReLU(),
            nn.Conv1d(cfg.D_MODEL, cfg.D_MODEL, 3, padding=1)
        )

        self.head = nn.Linear(cfg.D_MODEL, 1)

    def forward(self, x):

        B, T, H, W, C = x.shape

        x = x.permute(0,1,4,2,3)

        features = []

        for t in range(T):

            feat = self.spatial(x[:, t])

            features.append(feat)

        x = torch.stack(features, dim=1)

        x = self.transformer(x)

        x = x.permute(0,2,1)

        x = self.temporal_conv(x)

        x = x.permute(0,2,1)

        x = self.head(x)

        return x.squeeze(-1)


## Loss Functions

In [15]:

def negative_pearson(preds, labels):

    preds = preds - torch.mean(preds, dim=1, keepdim=True)
    labels = labels - torch.mean(labels, dim=1, keepdim=True)

    numerator = torch.sum(preds * labels, dim=1)

    denominator = torch.sqrt(
        torch.sum(preds**2, dim=1) *
        torch.sum(labels**2, dim=1)
    )

    loss = 1 - numerator / (denominator + 1e-8)

    return loss.mean()


In [16]:

def spectral_loss(pred, target):

    pred_fft = torch.fft.rfft(pred, dim=1)
    gt_fft = torch.fft.rfft(target, dim=1)

    pred_mag = torch.abs(pred_fft)
    gt_mag = torch.abs(gt_fft)

    return torch.mean(torch.abs(pred_mag - gt_mag))


## Training Setup

In [17]:

model = RPPGTransformer().to(cfg.DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.LR,
    weight_decay=1e-2,
    betas=(0.9, 0.95)
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.EPOCHS
)


## Training Loop

In [18]:

def train_one_epoch(loader):

    model.train()

    total_loss = 0

    for frames, signal in tqdm(loader):

        frames = frames.to(cfg.DEVICE)
        signal = signal.to(cfg.DEVICE)

        optimizer.zero_grad()

        pred = model(frames)

        min_len = min(pred.shape[1], signal.shape[1])

        pred = pred[:, :min_len]
        signal = signal[:, :min_len]

        pearson = negative_pearson(pred, signal)

        mse = F.smooth_l1_loss(pred, signal)

        spec = spectral_loss(pred, signal)

        loss = 0.5 * pearson + 0.3 * mse + 0.2 * spec

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    return total_loss / len(loader)


## Example Dataset Paths

In [20]:

video_paths = [
    r"C:\Users\UIET\Desktop\physformer\dataset"
]

dataset = RPPGDataset(video_paths)

loader = DataLoader(
    dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True
)


## Start Training

In [21]:

for epoch in range(cfg.EPOCHS):

    loss = train_one_epoch(loader)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f}")

torch.save(
    model.state_dict(),
    "advanced_rppg_transformer.pth"
)

print("Model Saved")


  0%|                                                                                            | 0/1 [00:01<?, ?it/s]


AxisError: axis 1 is out of bounds for array of dimension 1


# Recommended Future Improvements

## Best Next Upgrades

- PhysFormer
- EfficientPhys
- TS-CAN
- Motion augmentation
- Skin segmentation
- Attention pooling
- Multi-scale transformers
- Frequency-domain supervision
- Self-supervised pretraining

Expected MAE after improvements:
- ~2–5 BPM depending on dataset quality
